# Libs

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plot Style

In [2]:
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#e5e5e0",
    "grid.linewidth": 0.8,
    "axes.edgecolor": "#52514e",
    "text.color": "#0b0b0b",
    "axes.labelcolor": "#0b0b0b",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

COLOR_EXISTING = "#2a78d6"  # categorical slot 1 (blue)
COLOR_NEW = "#eb6834"  # categorical slot 2 (orange)
COLOR_NEUTRAL = "#8a8a85"  # muted reference lines
SEQUENTIAL_BLUES = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab"]

# Helpers

In [3]:
def find_non_product_codes(df):
    """Find StockCodes that do not look like real products.

    Normal Online Retail II product codes start with a digit
    (e.g. "85048", "79323P"). Codes that start with a letter are
    operational codes such as postage or bank charges rather than
    products.

    Args:
        df: Transaction DataFrame with a product_id column.

    Returns:
        A sorted list of the anomalous product_id values.
    """
    codes = df["product_id"].unique()
    anomalous = [c for c in codes if not c[:1].isdigit()]
    return sorted(anomalous)


def flag_valid_transactions(df, non_product_codes):
    """Flag rows that count as a valid purchase (Section 7.2).

    A valid purchase keeps a Customer ID, is not a cancellation
    (Invoice starting with "C"), has Quantity > 0 and Price > 0,
    is not a non-product stock code, and is not an exact duplicate
    of an earlier row.

    Args:
        df: Transaction DataFrame with renamed columns
            (invoice_id, customer_id, quantity, unit_price,
            product_id).
        non_product_codes: Collection of product_id values to
            exclude (e.g. from find_non_product_codes).

    Returns:
        A boolean Series aligned to df.index, True for valid rows.
    """
    has_customer = df["customer_id"].notna()
    not_cancelled = ~df["invoice_id"].str.startswith("C")
    valid_amounts = (df["quantity"] > 0) & (df["unit_price"] > 0)
    is_product = ~df["product_id"].isin(non_product_codes)
    not_duplicate = ~df.duplicated()
    return (
        has_customer
        & not_cancelled
        & valid_amounts
        & is_product
        & not_duplicate
    )

In [4]:
def compute_occasions(df):
    """Collapse valid transactions to one row per purchase occasion.

    A purchase occasion is one distinct invoice. Revenue and
    quantity are summed across the invoice's lines, the timestamp
    is the invoice's earliest line, and the distinct products
    bought are kept as a set (used for product-behavior features).

    Args:
        df: Valid transaction rows with customer_id, invoice_id,
            invoice_time, product_id, quantity and total_price
            columns.

    Returns:
        DataFrame with one row per (customer_id, invoice_id):
        customer_id, invoice_id, invoice_time, revenue, quantity,
        products (frozenset of product_id).
    """
    occasions = (
        df.groupby(["customer_id", "invoice_id"], as_index=False)
        .agg(
            invoice_time=("invoice_time", "min"),
            revenue=("total_price", "sum"),
            quantity=("quantity", "sum"),
            products=("product_id", lambda s: frozenset(s)),
        )
        .sort_values(["customer_id", "invoice_time"])
        .reset_index(drop=True)
    )
    return occasions

In [5]:
def build_snapshot_grid(occasions, snapshot_dates, r_max):
    """Build a Customer x Snapshot Date grid with recency.

    For every customer and every snapshot date, the most recent
    purchase on or before that date is looked up with a backward
    as-of merge. A customer with no purchase on or before the date
    is dropped (not yet an existing customer at that snapshot).

    Args:
        occasions: Purchase-occasion table with customer_id and
            invoice_time columns.
        snapshot_dates: Iterable of snapshot Timestamps (e.g. a
            weekly date range).
        r_max: Activity threshold in days; rows with
            recency_days <= r_max are flagged active.

    Returns:
        DataFrame with one row per (customer_id, snapshot_date) for
        customers with purchase history by that date: customer_id,
        snapshot_date, last_purchase_time, recency_days, is_active.
    """
    customers = pd.DataFrame(
        {"customer_id": occasions["customer_id"].unique()}
    )
    dates = pd.DataFrame({"snapshot_date": snapshot_dates})
    grid = customers.merge(dates, how="cross")
    grid = grid.sort_values("snapshot_date")

    history = occasions[["customer_id", "invoice_time"]].sort_values(
        "invoice_time"
    )
    grid = pd.merge_asof(
        grid,
        history,
        left_on="snapshot_date",
        right_on="invoice_time",
        by="customer_id",
        direction="backward",
    )
    grid = grid.dropna(subset=["invoice_time"]).rename(
        columns={"invoice_time": "last_purchase_time"}
    )
    recency = grid["snapshot_date"] - grid["last_purchase_time"]
    grid["recency_days"] = recency.dt.total_seconds() / 86400
    grid["is_active"] = grid["recency_days"] <= r_max
    return grid.sort_values(
        ["customer_id", "snapshot_date"]
    ).reset_index(drop=True)

In [6]:
def asof_lookup(targets, target_date_col, source, source_date_col,
                 value_cols):
    """Look up the most recent known value(s) as of each target date.

    For every (customer_id, target date) row in `targets`, finds the
    `source` row with the latest `source_date_col` on or before the
    target date for that customer, and attaches its `value_cols`.
    This is the single building block used throughout this notebook
    to compute point-in-time features without leaking information
    from after the target date.

    Args:
        targets: DataFrame with customer_id and target_date_col.
        target_date_col: Name of the date column in `targets`.
        source: DataFrame with customer_id, source_date_col, and
            every column listed in value_cols.
        source_date_col: Name of the date column in `source`.
        value_cols: List of column names in `source` to attach.

    Returns:
        A copy of `targets` with `value_cols` attached (NaN where no
        source row exists on or before the target date).
    """
    src = source[["customer_id", source_date_col] + value_cols]
    src = src.sort_values(source_date_col)
    result = pd.merge_asof(
        targets.sort_values(target_date_col),
        src,
        left_on=target_date_col,
        right_on=source_date_col,
        by="customer_id",
        direction="backward",
    )
    return result

In [7]:
def compute_cumulative_distinct_products(occasions):
    """Compute the running distinct-product count per occasion.

    For each customer, occasions are processed in time order while
    keeping a running set of products already seen; the count after
    each occasion is the number of distinct products bought up to
    and including that occasion.

    Args:
        occasions: Purchase-occasion table with customer_id,
            invoice_time and products (set-like) columns.

    Returns:
        A copy of `occasions`, sorted by (customer_id, invoice_time),
        with a new `distinct_products_so_far` column.
    """
    ordered = occasions.sort_values(
        ["customer_id", "invoice_time"]
    ).copy()
    counts = []
    for _, rows in ordered.groupby("customer_id"):
        seen = set()
        for products in rows["products"]:
            seen.update(products)
            counts.append(len(seen))
    ordered["distinct_products_so_far"] = counts
    return ordered

In [8]:
def add_churn_labels(grid, occasions, horizon, data_end):
    """Add the churn label to each labelable snapshot row.

    A row is labelable only if its full prediction window is
    observed, i.e. snapshot_date + horizon <= data_end. For those
    rows, a forward as-of merge finds the next purchase after the
    snapshot date; churn = 1 if no purchase falls within the
    (snapshot_date, snapshot_date + horizon] window.

    Args:
        grid: Output of build_snapshot_grid.
        occasions: Purchase-occasion table with customer_id and
            invoice_time columns.
        horizon: Prediction horizon in days (H).
        data_end: Timestamp of the last observed transaction.

    Returns:
        A copy of `grid` with two new columns: labelable (bool) and
        churn (0/1, NaN where not labelable).
    """
    window = pd.Timedelta(days=horizon)
    grid = grid.copy()
    grid["labelable"] = grid["snapshot_date"] + window <= data_end

    history = occasions[["customer_id", "invoice_time"]].sort_values(
        "invoice_time"
    )
    labelable_rows = grid[grid["labelable"]].sort_values(
        "snapshot_date"
    )
    labelable_rows = pd.merge_asof(
        labelable_rows,
        history,
        left_on="snapshot_date",
        right_on="invoice_time",
        by="customer_id",
        direction="forward",
        allow_exact_matches=False,
    )
    window_end = labelable_rows["snapshot_date"] + window
    no_purchase = labelable_rows["invoice_time"].isna()
    too_late = labelable_rows["invoice_time"] > window_end
    labelable_rows["churn"] = (no_purchase | too_late).astype(int)

    grid = grid.merge(
        labelable_rows[["customer_id", "snapshot_date", "churn"]],
        on=["customer_id", "snapshot_date"],
        how="left",
    )
    return grid

In [9]:
def apply_episode_cooldown(active_grid, cooldown_days):
    """Select one training snapshot per eligibility episode.

    Starting from a customer's first active snapshot date, the next
    training snapshot is the first later active date at least
    cooldown_days after it, and so on. This removes near-duplicate,
    overlapping snapshots from the training population.

    Args:
        active_grid: Active rows (is_active == True) from
            build_snapshot_grid, with customer_id and
            snapshot_date columns.
        cooldown_days: Minimum gap in days between two training
            snapshots of the same customer (C).

    Returns:
        The subset of active_grid rows selected as training
        snapshots.
    """
    cooldown = pd.Timedelta(days=cooldown_days)
    ordered = active_grid.sort_values(["customer_id", "snapshot_date"])
    selected_index = []
    for _, rows in ordered.groupby("customer_id"):
        next_allowed = None
        for row in rows.itertuples():
            allowed = next_allowed is None or row.snapshot_date >= (
                next_allowed
            )
            if allowed:
                selected_index.append(row.Index)
                next_allowed = row.snapshot_date + cooldown
    return active_grid.loc[selected_index]

In [10]:
def compute_embargoed_split(
    dates, horizon, target_val_weeks=8, target_test_weeks=10
):
    """Compute train/validation/test weekly date sets with embargo.

    Validation and test are sized to fixed operational targets and
    taken from the end of the labelable range working backward, with
    an embargo of `horizon` days before each; train absorbs whatever
    weeks remain before that (same allocation used in the Chapter 6
    feasibility check, now returning actual dates instead of counts
    so rows can be labeled with a split).

    Args:
        dates: Sorted DatetimeIndex of labelable weekly dates.
        horizon: Embargo length in days (H).
        target_val_weeks: Desired number of validation weeks.
        target_test_weeks: Desired number of test weeks.

    Returns:
        A dict with "train", "val" and "test" DatetimeIndex values.
    """
    embargo = pd.Timedelta(days=horizon)

    n_test = min(target_test_weeks, len(dates))
    test_dates = dates[-n_test:] if n_test else dates[:0]

    before_test = (
        dates[dates <= test_dates[0] - embargo]
        if len(test_dates)
        else dates
    )
    n_val = min(target_val_weeks, len(before_test))
    val_dates = before_test[-n_val:] if n_val else before_test[:0]

    train_dates = (
        before_test[before_test <= val_dates[0] - embargo]
        if len(val_dates)
        else before_test
    )
    return {"train": train_dates, "val": val_dates, "test": test_dates}

# Load

In [11]:
if os.path.exists("../data/raw/online_retail_II.parquet"):
    df_retail = pd.read_parquet("../data/raw/online_retail_II.parquet")
else:
    df_2009_2010 = pd.read_excel(
        "../data/raw/online_retail_II.xlsx",
        sheet_name="Year 2009-2010",
    )
    df_2010_2011 = pd.read_excel(
        "../data/raw/online_retail_II.xlsx",
        sheet_name="Year 2010-2011",
    )
    df_retail = pd.concat(
        [df_2009_2010, df_2010_2011], ignore_index=True
    )
    df_retail["Invoice"] = df_retail["Invoice"].astype(str)
    df_retail["StockCode"] = df_retail["StockCode"].astype(str)
    df_retail["Description"] = df_retail["Description"].astype(str)
    df_retail.to_parquet(
        "../data/raw/online_retail_II.parquet", index=False
    )

In [12]:
RENAME_COLUMNS = {
    "Invoice": "invoice_id",
    "StockCode": "product_id",
    "Description": "product_description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_time",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
}
df_retail = df_retail.rename(columns=RENAME_COLUMNS)
df_retail["invoice_time"] = pd.to_datetime(df_retail["invoice_time"])
df_retail["total_price"] = df_retail["quantity"] * df_retail["unit_price"]
df_retail.shape

(1067371, 9)

**Chosen parameters.** Inherited from the Data Understanding
notebook's Key Findings (Chapter 6, Section 6.14/6.15): prediction
horizon $H = 90$ days, activity threshold $R_{\max} = $ P90 of the
inter-purchase gap distribution ($\approx$136 days), training
episode cooldown $C = H = 90$ days. $H=90$ was the best-supported
candidate (24 training weeks after the embargoed split, matching
this chapter's own illustrative timeline), unlike $H=120$ which
left only 10 training weeks.

In [13]:
H = 90
C = H
R_MAX = 136.0
WARMUP = pd.DateOffset(months=6)

## 7.1 Data Cleaning

Clean transaction records using documented, reproducible rules
(Section 7.2). Every rule's row impact was already measured in the
Data Understanding notebook (Section 6.3); here they are combined
into the single `is_valid` filter used to build everything else in
this chapter.

In [14]:
non_product_codes = find_non_product_codes(df_retail)
is_valid = flag_valid_transactions(df_retail, non_product_codes)
df_valid = df_retail[is_valid].copy()
print(
    f"Valid rows kept: {len(df_valid):,} "
    f"({len(df_valid) / len(df_retail):.1%} of raw rows)"
)

Valid rows kept: 776,577 (72.8% of raw rows)


## 7.2 Valid Transaction Definition

A **valid purchase** row satisfies all of: Customer ID present, not
a cancellation (Invoice starting with "C"), Quantity > 0 and Price
> 0, not a non-product stock code, and not an exact duplicate. A
**purchase occasion** is one distinct invoice; revenue, quantity and
the set of products bought are aggregated to the occasion level.

In [15]:
occasions = compute_occasions(df_valid)
occasions = compute_cumulative_distinct_products(occasions)
occasions.head()

,customer_id,invoice_id,invoice_time,revenue,quantity,products,distinct_products_so_far
0,12346.0,499763,2010-03-02 13:08:00,27.05,5,"(20679, 20682, 15056BL, 15056P, 15056N)",5
1,12346.0,513774,2010-06-28 13:53:00,142.31,19,"(22687, 22689, 48111, 22365, 21955, 48173C, 48...",24
2,12346.0,541431,2011-01-18 10:01:00,77183.60,74215,(23166),25
3,12347.0,529924,2010-10-31 14:20:00,611.53,509,"(20985, 22331, 21578, 22418, 47599A, 22045, 84...",40
4,12347.0,537626,2010-12-07 14:57:00,711.79,319,"(22728, 22727, 84997D, 84969, 22772, 22375, 21...",70


## 7.3 Customer History Construction

Aggregate valid transaction history by customer while preserving
timestamps and product information -- this is exactly the
`occasions` table built above, which already carries per-occasion
revenue, quantity, products and the running distinct-product count
needed for point-in-time features in 7.6.

In [16]:
DATA_START = occasions["invoice_time"].min()
DATA_END = occasions["invoice_time"].max()
print(f"Customers with history: {occasions['customer_id'].nunique():,}")
print(f"Occasions: {len(occasions):,}")
print(f"History span: {DATA_START.date()} to {DATA_END.date()}")

Customers with history: 5,852
Occasions: 36,594
History span: 2009-12-01 to 2011-12-09


## 7.4 Eligibility Construction

**Step 1 -- weekly active-customer base.** At each weekly Monday
date $t$ (after the 6-month warm-up), a customer is active if they
have a valid purchase on or before $t$ (handled by
`build_snapshot_grid` dropping customers with no prior purchase) and
Recency at $t$ $\le R_{\max}$.

In [17]:
warmup_end = DATA_START + WARMUP
weekly_dates = pd.date_range(
    start=warmup_end, end=DATA_END, freq="W-MON"
)

base = build_snapshot_grid(occasions, weekly_dates, R_MAX)
base = base[base["is_active"]].copy()
print(f"Weekly active-customer base rows: {len(base):,}")
base.head()

Weekly active-customer base rows: 208,764


,customer_id,snapshot_date,last_purchase_time,recency_days,is_active
0,12346.0,2010-06-07 07:45:00,2010-03-02 13:08:00,96.775694,True
1,12346.0,2010-06-14 07:45:00,2010-03-02 13:08:00,103.775694,True
2,12346.0,2010-06-21 07:45:00,2010-03-02 13:08:00,110.775694,True
3,12346.0,2010-06-28 07:45:00,2010-03-02 13:08:00,117.775694,True
4,12346.0,2010-07-05 07:45:00,2010-06-28 13:53:00,6.744444,True


**Step 2 -- population flags on the base table.**
`in_training_sample` is set by the eligibility-episode rule (3.9);
`in_scoring_population` is every active row (the historical data
contain no real campaigns, so suppression is empty); `sample_weight`
needs the train period, so it is filled in once the embargoed split
is computed in 7.9.

In [18]:
episode_rows = apply_episode_cooldown(base, cooldown_days=C)
base["in_training_sample"] = base.index.isin(episode_rows.index)
base["in_scoring_population"] = base["is_active"]
base["sample_weight"] = np.nan

print(
    f"in_training_sample: {base['in_training_sample'].sum():,} rows "
    f"({base['in_training_sample'].mean():.1%} of base)"
)
print(
    f"in_scoring_population: {base['in_scoring_population'].sum():,} "
    "rows"
)

in_training_sample: 20,152 rows (9.7% of base)
in_scoring_population: 208,764 rows


## 7.5 Customer Snapshot Construction

`base` above **is** the one weekly active-customer base table of
Customer $\times$ Snapshot Date rows. The Training Population and
Scoring Population are not separate tables -- they are the
`in_training_sample` and `in_scoring_population` flags on this same
table, which guarantees both use identical feature and label
definitions. Sanity check: every training-sample row must also be
an active, scoring-population row (the episode rule only subsets
the active base, never adds to it).

In [19]:
assert (
    base.loc[base["in_training_sample"], "in_scoring_population"]
).all(), "Training-sample rows must be a subset of active rows"
base.shape

(208764, 8)

## 7.6 Feature Engineering

Generate historical features using information available **strictly
before or at** the snapshot date only, via the `asof_lookup` helper
(a backward as-of merge) on cumulative, per-occasion statistics --
the same pattern used for recency in `build_snapshot_grid`, so every
feature is leakage-safe by construction. A representative subset of
each group from the Feature Engineering Strategy (Section 3.11) is
implemented: RFM (recency, frequency, monetary), purchase behavior
(average order value), temporal behavior (tenure, mean inter-purchase
gap, recency relative to that gap, activity in the last 90 days),
product behavior (distinct products so far), and cancellation
behavior (cancellation count and value share so far).

In [20]:
occasion_stats = occasions.sort_values(
    ["customer_id", "invoice_time"]
).copy()
occasion_stats["frequency_so_far"] = (
    occasion_stats.groupby("customer_id").cumcount() + 1
)
occasion_stats["monetary_so_far"] = occasion_stats.groupby(
    "customer_id"
)["revenue"].cumsum()
occasion_stats["quantity_so_far"] = occasion_stats.groupby(
    "customer_id"
)["quantity"].cumsum()

gap_days = (
    occasion_stats.groupby("customer_id")["invoice_time"]
    .diff()
    .dt.total_seconds()
    / 86400
)
occasion_stats["mean_gap_so_far"] = gap_days.groupby(
    occasion_stats["customer_id"]
).transform(lambda s: s.expanding().mean())

occasion_stats[
    ["customer_id", "invoice_time", "frequency_so_far",
     "monetary_so_far", "mean_gap_so_far"]
].head()

,customer_id,invoice_time,frequency_so_far,monetary_so_far,mean_gap_so_far
0,12346.0,2010-03-02 13:08:00,1,27.05,NaN
1,12346.0,2010-06-28 13:53:00,2,169.36,118.031250
2,12346.0,2011-01-18 10:01:00,3,77352.96,160.935069
3,12347.0,2010-10-31 14:20:00,1,611.53,NaN
4,12347.0,2010-12-07 14:57:00,2,1323.32,37.025694


In [ ]:
feature_cols = [
    "frequency_so_far", "monetary_so_far", "quantity_so_far",
    "mean_gap_so_far", "distinct_products_so_far",
]
base = asof_lookup(
    base, "snapshot_date", occasion_stats, "invoice_time", feature_cols
)
base = base.drop(columns=["invoice_time"])

base["avg_order_value_so_far"] = (
    base["monetary_so_far"] / base["frequency_so_far"]
)
raw_ratio = base["recency_days"] / base["mean_gap_so_far"]
n_degenerate = np.isinf(raw_ratio).sum()
base["recency_to_typical_gap"] = raw_ratio.replace(
    [np.inf, -np.inf], np.nan
)
print(
    f"recency_to_typical_gap: {n_degenerate} rows had a zero "
    "mean_gap_so_far (same-instant repeat purchases) and were set "
    "to NaN instead of inf"
)
base[feature_cols + ["avg_order_value_so_far"]].describe().round(2)

In [22]:
first_purchase = occasions.groupby("customer_id")[
    "invoice_time"
].min()
base = base.merge(
    first_purchase.rename("first_purchase_time"),
    left_on="customer_id",
    right_index=True,
    how="left",
)
base["tenure_days"] = (
    base["snapshot_date"] - base["first_purchase_time"]
).dt.total_seconds() / 86400
base[["tenure_days"]].describe().round(1)

,tenure_days
count,208764.0
mean,277.9
std,191.9
min,0.6
25%,109.8
50%,248.8
75%,420.8
max,733.9


In [23]:
window_start = base[["customer_id", "snapshot_date"]].copy()
window_start["window_start_date"] = (
    window_start["snapshot_date"] - pd.Timedelta(days=90)
)
freq_at_window_start = asof_lookup(
    window_start, "window_start_date", occasion_stats, "invoice_time",
    ["frequency_so_far"],
)["frequency_so_far"].fillna(0)

base["activity_last_90d"] = (
    base["frequency_so_far"] - freq_at_window_start
).clip(lower=0)
base[["activity_last_90d"]].describe().round(2)

,activity_last_90d
count,208764.00
mean,4.17
std,11.00
min,0.00
25%,0.00
50%,1.00
75%,4.00
max,361.00


In [24]:
cancellations = df_retail[
    df_retail["invoice_id"].str.startswith("C")
    & df_retail["customer_id"].notna()
].copy()
cancel_occasions = (
    cancellations.groupby(["customer_id", "invoice_id"], as_index=False)
    .agg(invoice_time=("invoice_time", "min"),
         cancel_value=("total_price", "sum"))
    .sort_values(["customer_id", "invoice_time"])
)
cancel_occasions["cancel_value"] = cancel_occasions["cancel_value"].abs()
cancel_occasions["cancel_count_so_far"] = cancel_occasions.groupby(
    "customer_id"
).cumcount() + 1
cancel_occasions["cancel_value_so_far"] = cancel_occasions.groupby(
    "customer_id"
)["cancel_value"].cumsum()

base = asof_lookup(
    base, "snapshot_date", cancel_occasions, "invoice_time",
    ["cancel_count_so_far", "cancel_value_so_far"],
)
base = base.drop(columns=["invoice_time"])
base["cancel_count_so_far"] = base["cancel_count_so_far"].fillna(0)
base["cancel_value_so_far"] = base["cancel_value_so_far"].fillna(0)
base["cancel_value_share_so_far"] = base["cancel_value_so_far"] / (
    base["monetary_so_far"] + base["cancel_value_so_far"]
)
base[
    ["cancel_count_so_far", "cancel_value_share_so_far"]
].describe().round(3)

,cancel_count_so_far,cancel_value_share_so_far
count,208764.000,208764.000
mean,1.490,0.022
std,3.538,0.059
min,0.000,0.000
25%,0.000,0.000
50%,0.000,0.000
75%,2.000,0.020
max,112.000,0.896


## 7.7 Churn Label Construction

Use only the prediction window $(t, t+H]$ to determine churn,
applying the same valid-purchase definition as 7.2 (the `occasions`
table). Final (non-provisional) label at $H=90$ days.

In [25]:
base = add_churn_labels(base, occasions, H, DATA_END)
print(
    f"Labelable rows: {base['labelable'].sum():,} / {len(base):,}"
)
print(f"Churn rate (labelable rows): {base['churn'].mean():.1%}")

Labelable rows: 172,561 / 208,764
Churn rate (labelable rows): 48.0%


## 7.8 Leakage Prevention

No future information may be used in feature construction; the
prediction window is reserved for the label only. Every feature
above was attached with `asof_lookup`'s **backward** merge, which
structurally cannot see a source row dated after the target date --
verified explicitly below. $R_{\max}$ and $H$ themselves were
chosen from inter-purchase gaps measured over the **full** dataset
in Chapter 6, not the training period alone; a stricter pipeline
would re-derive them from training-period data only. Aggregates that
depend on the whole dataset (scalers, encoders, imputation values)
are deferred to Chapter 8, fitted on the training period only once
modeling starts.

In [26]:
check = asof_lookup(
    base[["customer_id", "snapshot_date"]], "snapshot_date",
    occasion_stats, "invoice_time", ["frequency_so_far"],
)
assert (check["invoice_time"] <= check["snapshot_date"]).all(), (
    "Found a feature sourced from after its snapshot date"
)
print("OK: every attached feature's source date is <= snapshot_date")

OK: every attached feature's source date is <= snapshot_date


## 7.9 Temporal Dataset Split with Embargo

Split chronologically by snapshot date into train, validation and
test, with an **embargo of $H$ days** between consecutive periods,
so that labels of late training snapshots are never observed during
validation. Validation and test are sized to 8 and 10 weeks (the
`compute_embargoed_split` helper); train absorbs the remaining
labelable weeks.

In [27]:
labelable_dates = weekly_dates[
    weekly_dates + pd.Timedelta(days=H) <= DATA_END
]
split_dates = compute_embargoed_split(labelable_dates, H)

split = pd.Series(index=base.index, dtype=object)
split[base["snapshot_date"].isin(split_dates["train"])] = "train"
split[base["snapshot_date"].isin(split_dates["val"])] = "validation"
split[base["snapshot_date"].isin(split_dates["test"])] = "test"
base["split"] = split

for name, dates in split_dates.items():
    if len(dates):
        print(
            f"{name}: {len(dates)} weeks, "
            f"{dates.min().date()} to {dates.max().date()}"
        )
base["split"].value_counts(dropna=False)

train: 24 weeks, 2010-06-07 to 2010-11-15
val: 8 weeks, 2011-02-14 to 2011-04-04
test: 10 weeks, 2011-07-04 to 2011-09-05


split
NaN           101583
train          61415
test           24252
validation     21514
Name: count, dtype: int64

In [28]:
train_active = base[
    (base["split"] == "train") & base["is_active"]
]
train_counts_per_customer = train_active.groupby(
    "customer_id"
).size()
weight = 1 / train_counts_per_customer
base.loc[train_active.index, "sample_weight"] = base.loc[
    train_active.index, "customer_id"
].map(weight)
base.loc[train_active.index, "sample_weight"].describe().round(4)

count    61415.0000
mean         0.0636
std          0.0619
min          0.0417
25%          0.0417
50%          0.0435
75%          0.0588
max          1.0000
Name: sample_weight, dtype: float64

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))

segments = [
    ("Train", split_dates["train"].min(), split_dates["train"].max(),
     COLOR_EXISTING),
    ("Embargo", split_dates["train"].max(), split_dates["val"].min(),
     COLOR_NEUTRAL),
    ("Validation", split_dates["val"].min(), split_dates["val"].max(),
     COLOR_EXISTING),
    ("Embargo", split_dates["val"].max(), split_dates["test"].min(),
     COLOR_NEUTRAL),
    ("Test", split_dates["test"].min(), split_dates["test"].max(),
     COLOR_EXISTING),
]
for label, start, end, color in segments:
    width = (end - start).days + 7
    ax.barh(
        0, width, left=start, height=0.5, color=color,
        edgecolor="white",
    )
    ax.annotate(
        label, (start + (end - start) / 2, 0), ha="center",
        va="center", color="white", fontsize=9,
    )

ax.set_yticks([])
ax.set_xlabel("Snapshot date")
ax.set_title(
    "Embargoed train / validation / test split "
    f"(H={H} days, embargo={H} days)"
)
fig.tight_layout()
plt.show()

## 7.10 Final Modeling Dataset

Each row represents **Customer $\times$ Snapshot Date**, with a
segment flag (one-time / repeat, based on frequency so far),
population flags, sample weight, historical features, the churn
label (where labelable) and the split assignment -- produced by the
reproducible, leakage-checked transformations above from the single
configuration (`H`, `R_MAX`, `C`) set at the top of this notebook.

In [29]:
base["segment"] = np.where(
    base["frequency_so_far"] == 1, "one-time", "repeat"
)

modeling_dataset = base[[
    "customer_id", "snapshot_date", "segment",
    "in_training_sample", "in_scoring_population", "sample_weight",
    "recency_days", "frequency_so_far", "monetary_so_far",
    "avg_order_value_so_far", "tenure_days", "mean_gap_so_far",
    "recency_to_typical_gap", "activity_last_90d",
    "distinct_products_so_far", "cancel_count_so_far",
    "cancel_value_share_so_far", "labelable", "churn", "split",
]].reset_index(drop=True)

print(modeling_dataset.shape)
modeling_dataset.head()

(208764, 20)


,customer_id,snapshot_date,segment,in_training_sample,in_scoring_population,sample_weight,recency_days,frequency_so_far,monetary_so_far,avg_order_value_so_far,tenure_days,mean_gap_so_far,recency_to_typical_gap,activity_last_90d,distinct_products_so_far,cancel_count_so_far,cancel_value_share_so_far,labelable,churn,split
0,12346.0,2010-06-07 07:45:00,one-time,True,True,0.043478,96.775694,1,27.05,27.050000,96.775694,NaN,NaN,0.0,5,1.0,0.792800,True,0.0,train
1,17074.0,2010-06-07 07:45:00,one-time,True,True,0.055556,11.820139,1,203.05,203.050000,11.820139,NaN,NaN,0.0,4,0.0,0.000000,True,1.0,train
2,13854.0,2010-06-07 07:45:00,repeat,True,True,0.041667,24.822917,7,1785.47,255.067143,149.845139,20.837037,1.191288,7.0,27,2.0,0.006261,True,0.0,train
3,13834.0,2010-06-07 07:45:00,repeat,True,True,0.062500,25.887500,2,324.19,162.095000,118.786806,92.899306,0.278662,0.0,15,1.0,0.190517,True,1.0,train
4,17075.0,2010-06-07 07:45:00,repeat,True,True,0.047619,81.633333,4,758.08,189.520000,174.847917,31.071528,2.627271,3.0,56,0.0,0.000000,True,0.0,train


In [30]:
os.makedirs("../data/processed", exist_ok=True)
out_path = f"../data/processed/modeling_dataset_h{H}.parquet"
modeling_dataset.to_parquet(out_path, index=False)
print(f"Saved {len(modeling_dataset):,} rows to {out_path}")

Saved 208,764 rows to ../data/processed/modeling_dataset_h90.parquet


In [ ]:
modeling_dataset.head()

In [31]:
modeling_dataset.groupby("split", dropna=False).agg(
    rows=("customer_id", "size"),
    customers=("customer_id", "nunique"),
    churn_rate=("churn", "mean"),
).round(3)

,rows,customers,churn_rate
split,,,
test,24252,2907,0.401
train,61415,3905,0.430
validation,21514,3328,0.529
NaN,101583,5209,0.540
